# 02 — Schema Validation (Canonical Contracts)

**Fecha:** 2026-02-02  
**Objetivo:** Definir y validar el *schema canonical* para OHLCV 1m y Quotes (p95) usando el slice AABA 2019-01.

**Inputs:**
- Manifest: `data/manifests/r2_slice_AABA_2019_01.json`
- Cache local: `data/cache/...`

**Outputs:**
- Contrato de schema (documentado en este notebook)
- Métricas de calidad específicas del schema (crossed markets, duplicados, nulos)
- Reglas/decisiones para normalización y tratamiento


In [9]:
import sys, os, time
print("python:", sys.executable)
print("cwd:", os.getcwd())
t0 = time.time()
time.sleep(0.2)
print("ok, elapsed:", round(time.time() - t0, 3))

python: /Users/alex/Desktop/TSIS.AI/Backtesting_system/backtest/bin/python
cwd: /Users/alex/Desktop/TSIS.AI/Backtesting_system
ok, elapsed: 0.203


In [10]:
import json
from pathlib import Path
import polars as pl

from src.core.settings import settings

MANIFEST_PATH = Path("data/manifests/r2_slice_AABA_2019_01.json")
CACHE_DIR = Path(settings.DATA_CACHE_DIR)

print("Manifest:", MANIFEST_PATH)
print("Cache dir:", CACHE_DIR)


Manifest: data/manifests/r2_slice_AABA_2019_01.json
Cache dir: data/cache


In [11]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
objs = manifest["objects"]

ohlcv_keys = [o["key"] for o in objs if o["dataset"] == "ohlcv_intraday_1m"]
quote_keys = [o["key"] for o in objs if o["dataset"] == "quotes_p95"]

print("OHLCV files:", len(ohlcv_keys))
print("Quote files:", len(quote_keys))
print("Example OHLCV:", ohlcv_keys[0])
print("Example Quote:", quote_keys[0])


OHLCV files: 1
Quote files: 21
Example OHLCV: ohlcv_intraday_1m/2019_2025/AABA/year=2019/month=01/minute.parquet
Example Quote: quotes_p95/AABA/year=2019/month=01/day=02/quotes.parquet


In [12]:
ohlcv_path = CACHE_DIR / ohlcv_keys[0]
df_ohlcv = pl.read_parquet(ohlcv_path, n_rows=5)

print("OHLCV columns:", df_ohlcv.columns)
print("OHLCV dtypes:", df_ohlcv.dtypes)
df_ohlcv


OHLCV columns: ['volume', 'vwap', 'open', 'close', 'high', 'low', 't', 'transactions', 'timestamp', 'ticker', 'date', 'minute']
OHLCV dtypes: [Int64, Float64, Float64, Float64, Float64, Float64, Int64, Int64, Datetime(time_unit='ms', time_zone=None), String, Date, String]


volume,vwap,open,close,high,low,t,transactions,timestamp,ticker,date,minute
i64,f64,f64,f64,f64,f64,i64,i64,datetime[ms],str,date,str
1000,57.0,57.0,57.0,57.0,57.0,1546435560000,5,2019-01-02 13:26:00,"""AABA""",2019-01-02,"""2019-01-02 13:26"""
188,57.16,57.16,57.16,57.16,57.16,1546436820000,1,2019-01-02 13:47:00,"""AABA""",2019-01-02,"""2019-01-02 13:47"""
11114,57.94,57.94,57.94,57.94,57.94,1546437000000,1,2019-01-02 13:50:00,"""AABA""",2019-01-02,"""2019-01-02 13:50"""
22833,56.8276,56.78,56.72,56.97,56.72,1546439400000,111,2019-01-02 14:30:00,"""AABA""",2019-01-02,"""2019-01-02 14:30"""
21724,56.7218,56.74,56.59,56.84,56.57,1546439460000,159,2019-01-02 14:31:00,"""AABA""",2019-01-02,"""2019-01-02 14:31"""


In [13]:
quote_path = CACHE_DIR / quote_keys[0]
df_q = pl.read_parquet(quote_path, n_rows=5)

print("Quotes columns:", df_q.columns)
print("Quotes dtypes:", df_q.dtypes)
df_q


Quotes columns: ['ask_exchange', 'ask_price', 'ask_size', 'bid_exchange', 'bid_price', 'bid_size', 'conditions', 'participant_timestamp', 'sequence_number', 'timestamp', 'tape', 'indicators']
Quotes dtypes: [Int64, Float64, Int64, Int64, Float64, Int64, List(Int64), Int64, Int64, Int64, Int64, List(Int64)]


ask_exchange,ask_price,ask_size,bid_exchange,bid_price,bid_size,conditions,participant_timestamp,sequence_number,timestamp,tape,indicators
i64,f64,i64,i64,f64,i64,list[i64],i64,i64,i64,i64,list[i64]
18,57.17,200,15,56.33,700,[1],1546439400000189852,303792,1546439400060097104,3,null
12,57.17,1000,15,56.33,700,[1],1546439400297882330,305701,1546439400297899484,3,null
12,57.17,1000,2,56.38,300,[1],1546439400319048147,306018,1546439400319064390,3,null
12,57.17,2000,2,56.38,300,[1],1546439400463833200,307975,1546439400463852123,3,null
7,57.17,5200,2,56.38,300,[1],1546439400685353000,310328,1546439400685553615,3,null


## Canonical schema contract

### OHLCV 1m (canonical)
- ts: int64 (epoch ns/us/ms) o datetime64[ns] (definir política)
- open, high, low, close: float64
- volume: int64

### Quotes p95 (canonical)
- ts: int64 (epoch …) tomado de `timestamp` (por defecto)
- bid: float64 (desde bid_price)
- ask: float64 (desde ask_price)
- bid_size, ask_size: int64
- (otros campos se conservan como metadata, no requeridos para checks básicos)


**Normalización en notebook: bid/ask/ts**

In [17]:
def normalize_quotes(df: pl.DataFrame) -> pl.DataFrame:
    out = df
    if "bid" not in out.columns and "bid_price" in out.columns:
        out = out.rename({"bid_price": "bid"})
    if "ask" not in out.columns and "ask_price" in out.columns:
        out = out.rename({"ask_price": "ask"})
    # canonical timestamp column name
    if "ts" not in out.columns and "timestamp" in out.columns:
        out = out.rename({"timestamp": "ts"})
    return out

df_qn = normalize_quotes(pl.read_parquet(quote_path))

[i for i in df_qn.columns]

['ask_exchange',
 'ask',
 'ask_size',
 'bid_exchange',
 'bid',
 'bid_size',
 'conditions',
 'participant_timestamp',
 'sequence_number',
 'ts',
 'tape',
 'indicators']

**Medir crossed markets por día**

Esto cuantifica el WARN:

In [18]:
from datetime import datetime

def crossed_ratio_for_file(key: str) -> dict:
    p = CACHE_DIR / key
    df = normalize_quotes(pl.read_parquet(p, columns=["bid_price","ask_price","timestamp"]))
    # after normalize, columns are bid/ask/ts
    crossed = df.filter(pl.col("bid") > pl.col("ask")).height
    total = df.height
    return {
        "key": key,
        "rows": total,
        "crossed": crossed,
        "crossed_ratio": (crossed / total) if total else 0.0,
    }

rows = [crossed_ratio_for_file(k) for k in quote_keys]
df_cross = pl.DataFrame(rows).with_columns([
    (pl.col("crossed_ratio") * 1e6).alias("crossed_ppm")  # parts-per-million
])

df_cross.sort("crossed_ratio", descending=True).head(10)


key,rows,crossed,crossed_ratio,crossed_ppm
str,i64,i64,f64,f64
"""quotes_p95/AABA/year=2019/mont…",293436,122,0.000416,415.763574
"""quotes_p95/AABA/year=2019/mont…",119021,17,0.000143,142.831937
"""quotes_p95/AABA/year=2019/mont…",127937,9,0.00007,70.347124
"""quotes_p95/AABA/year=2019/mont…",158881,11,0.000069,69.234207
"""quotes_p95/AABA/year=2019/mont…",212027,14,0.000066,66.029326
"""quotes_p95/AABA/year=2019/mont…",173296,11,0.000063,63.47521
"""quotes_p95/AABA/year=2019/mont…",190022,8,0.000042,42.100388
"""quotes_p95/AABA/year=2019/mont…",50000,2,0.00004,40.0
"""quotes_p95/AABA/year=2019/mont…",159598,6,0.000038,37.594456


In [19]:
# Resumen global)
df_cross.select([
    pl.len().alias("files"),
    pl.sum("rows").alias("total_rows"),
    pl.sum("crossed").alias("total_crossed"),
    (pl.sum("crossed") / pl.sum("rows")).alias("crossed_ratio_global"),
    (pl.sum("crossed") / pl.sum("rows") * 1e6).alias("crossed_ppm_global"),
])


files,total_rows,total_crossed,crossed_ratio_global,crossed_ppm_global
u32,i64,i64,f64,f64
21,3359402,229,0.000068,68.166894


## Decisión: tratamiento de crossed markets

Se observa `bid > ask` en el slice (AABA 2019-01) con ratio global medido.

Política (v1):
- Mantener como WARN en validación.
- Para backtesting: aplicar “clamp” en el modelo de ejecución:
  - bid = min(bid, ask)
  - ask = max(ask, bid)
  - y registrar cuántas filas fueron corregidas.


**Guardar resultados en runs para trazabilidad**

In [ ]:
OUT = Path("runs/data_quality/AABA_2019_01/schema_review")
OUT.mkdir(parents=True, exist_ok=True)

df_cross.write_parquet(OUT / "crossed_by_file.parquet")
print("Saved:", OUT / "crossed_by_file.parquet")

Saved: runs/data_quality/AABA_2019_01/schema_review/crossed_by_file.parquet
